# PMAP-Net training

In Colab, mount Drive and run the cells below. The default 4×4 BPSK run uses the saved `z` and `R` values to predict the local LLRs logged by Partial MAP. Place `layer_0.csv` through `layer_3.csv` or `rayleigh_pmap_layers.zip` in `MyDrive/EE_MIMO_PROJECT/rayleigh_data_4_by_4_BPSK/`.

Each run saves the model checkpoint and per-layer loss figure in a timestamped Drive folder.

In [ ]:
# @title Mount Drive and load the trainer
from pathlib import Path
import sys

from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/EE_MIMO_PROJECT")
trainer_path = PROJECT_ROOT / "src/models/train_pmapnet.py"
if not trainer_path.is_file():
    raise FileNotFoundError(
        f"Could not find the trainer at {trainer_path}. "
        "Set PROJECT_ROOT to the folder containing the project src directory."
    )

sys.path.insert(0, str(PROJECT_ROOT))
from src.models.train_pmapnet import main as train_pmapnet

In [ ]:
# @title Training settings
MODULATION = "bpsk"
NUM_TX = 4
HIDDEN_LAYERS = 2
EPOCHS = 100
BATCH_SIZE = 100
TRAIN_SAMPLES = 20_000
LEARNING_RATE = 0.005
SEED = 42
DEVICE = "auto"  # "auto", "cuda", or "cpu"

LAYER_DATA = "rayleigh_data_4_by_4_BPSK"  # Relative to PROJECT_ROOT
OUTPUT_DIR = "/content/drive/MyDrive/EE_MIMO_PROJECT/02_colab_output/"

In [ ]:
# @title Train PMAP-Net
arguments = [
    "--modulation", MODULATION,
    "--num-tx", str(NUM_TX),
    "--hidden-layers", str(HIDDEN_LAYERS),
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--train-samples", str(TRAIN_SAMPLES),
    "--learning-rate", str(LEARNING_RATE),
    "--seed", str(SEED),
    "--output-dir", OUTPUT_DIR,
    "--layer-data", str(PROJECT_ROOT / LAYER_DATA),
]
if DEVICE != "auto":
    arguments.extend(["--device", DEVICE])

run_dir = train_pmapnet(arguments)

In [ ]:
# @title Show the loss curves and saved files
from IPython.display import Image, display

display(Image(filename=str(run_dir / "layer_loss.png")))
print(f"Saved in {run_dir}")
for checkpoint in sorted(run_dir.glob("pmapnet_*.pt")):
    print(checkpoint.name)